# Explainable Drug–Disease Relation Extraction from Biomedical Text
### A Comparative Study: Pipeline vs Joint Models on BC5CDR — Enhanced with LLM + RAG

**Course:** 22AIE315 — Natural Language Processing | **Term Project**

---

## Architecture Overview

```
PubMed Text
    │
    ├──► BioBERT NER ──────────────► [Chemical, Disease] entities
    │                                          │
    ├──► BioBERT RE ───────────────► CID relation + confidence score
    │                                          │
    ├──► ChromaDB RAG ─────────────► Top-k supporting evidence chunks
    │         ▲                                │
    │    (indexed from                         ▼
    │  uploaded documents)       LLM (Claude API / OpenAI)
    │                                          │
    └──────────────────────────────► Clinical explanation + NER verification
```

**This notebook adds on top of the original Pipeline/Joint model comparison:**
- ChromaDB vector store built from your own documents (PDFs, text files, abstracts)
- RAG pipeline that retrieves supporting evidence for each predicted drug–disease pair
- LLM API (Anthropic Claude or OpenAI GPT-4) for clinical explanation generation
- LLM-based NER verification to catch BioBERT errors on rare drug names
- Combined end-to-end demo with all components wired together

**Runtime:** Google Colab with T4 GPU recommended.

## 0. Installation

In [ ]:
!pip install google-generativeai -q

In [ ]:
# Core NLP + ML
!pip install datasets transformers torch scikit-learn seqeval shap lime matplotlib pandas numpy -q

# RAG + Vector DB
!pip install chromadb sentence-transformers -q

# LLM API clients
!pip install anthropic openai -q

# Document loaders (for embedding your own docs)
!pip install pypdf python-docx -q

print("All packages installed.")

## 1. Configuration — API Keys & Settings

**Insert your API key below.** You can use either Anthropic (Claude) or OpenAI (GPT-4). Set `LLM_PROVIDER` accordingly.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  ██████╗ ██████╗ ███╗   ██╗███████╗██╗ ██████╗
# ██╔════╝██╔═══██╗████╗  ██║██╔════╝██║██╔════╝
# ██║     ██║   ██║██╔██╗ ██║█████╗  ██║██║  ███╗
# ██║     ██║   ██║██║╚██╗██║██╔══╝  ██║██║   ██║
# ╚██████╗╚██████╔╝██║ ╚████║██║     ██║╚██████╔╝
#  ╚═════╝ ╚═════╝ ╚═╝  ╚═══╝╚═╝     ╚═╝ ╚═════╝
# ═══════════════════════════════════════════════════════════════════════════

# ─── STEP 1: Choose your LLM provider ────────────────────────────────────
LLM_PROVIDER = "gemini"   # Options: "anthropic"  |  "openai"

# ─── STEP 2: Paste your API key here ─────────────────────────────────────
ANTHROPIC_API_KEY = ""  # ← Anthropic key
OPENAI_API_KEY    = ""       # ← OpenAI key (if using)
GEMINI_API_KEY = ""   # ← paste your key here
GEMINI_MODEL = "gemini-2.0-flash"
# ─── STEP 3: Choose LLM model ────────────────────────────────────────────
ANTHROPIC_MODEL = "claude-opus-4-6-20250514"  # Anthropic model to use
OPENAI_MODEL    = "gpt-4o"                  # OpenAI model to use

# ─── STEP 4: ChromaDB settings ───────────────────────────────────────────
CHROMA_PERSIST_DIR  = "/content/chroma_db"  # Where ChromaDB stores vectors on disk
CHROMA_COLLECTION   = "biomedical_docs"      # Collection name
EMBEDDING_MODEL     = "all-MiniLM-L6-v2"    # SentenceTransformer model for embeddings
CHUNK_SIZE          = 400                    # Characters per chunk when splitting documents
CHUNK_OVERLAP       = 80                     # Overlap between consecutive chunks
TOP_K_RETRIEVAL     = 3                      # How many chunks to retrieve per query

# ─── STEP 5: BioBERT + training settings ─────────────────────────────────
MODEL_NAME  = "dmis-lab/biobert-base-cased-v1.2"
BASE_PATH   = "/content/"                    # Folder containing PubTator .txt files

# ═══════════════════════════════════════════════════════════════════════════
print(f"LLM Provider : {LLM_PROVIDER.upper()}")
print(f"LLM Model    : {ANTHROPIC_MODEL if LLM_PROVIDER == 'anthropic' else OPENAI_MODEL}")
print(f"Embedding    : {EMBEDDING_MODEL}")
print(f"ChromaDB dir : {CHROMA_PERSIST_DIR}")
print("Config loaded.")

## 2. Imports

In [ ]:
import os, sys, re, json, warnings
import torch
import google.generativeai as genai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
from pathlib import Path

# PyTorch / HuggingFace
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn as nn
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
    pipeline as hf_pipeline
)

# Metrics
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
from seqeval.metrics import (
    f1_score as seqeval_f1,
    precision_score as seqeval_precision,
    recall_score as seqeval_recall,
    classification_report as ner_report
)

# Explainability
import shap
from lime.lime_text import LimeTextExplainer

# Vector DB + Embeddings
import chromadb
from sentence_transformers import SentenceTransformer

# LLM clients
import anthropic
import openai

# Document loaders
import pypdf
import docx as python_docx

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ─── Initialise LLM client ────────────────────────────────────────────────
if LLM_PROVIDER == "anthropic":
    llm_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
elif LLM_PROVIDER == "openai":
    llm_client = openai.OpenAI(api_key=OPENAI_API_KEY)
else:
    genai.configure(api_key=GEMINI_API_KEY)
    llm_client = genai.GenerativeModel(GEMINI_MODEL)
    print(f"LLM client  : Gemini ({GEMINI_MODEL})")

## 3. RAG — Document Ingestion & ChromaDB Vector Store

### How to add your own documents

Upload any of the following to `/content/rag_docs/` before running this section:
- `.txt` files (plain text, e.g. PubMed abstracts)
- `.pdf` files (biomedical papers, drug handbooks)
- `.docx` files (clinical notes, guidelines)
- The BC5CDR PubTator `.txt` files are **also automatically indexed** as part of the corpus

The cell below will:
1. Read and chunk all uploaded documents
2. Embed them with `sentence-transformers`
3. Store vectors in ChromaDB (persisted to `CHROMA_PERSIST_DIR`)

In [ ]:
# ─── Create document upload folder ────────────────────────────────────────
RAG_DOCS_DIR = "/content/rag_docs"
os.makedirs(RAG_DOCS_DIR, exist_ok=True)
os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

print("="*60)
print("DOCUMENT UPLOAD INSTRUCTIONS")
print("="*60)
print(f"Upload your documents to: {RAG_DOCS_DIR}")
print()
print("Supported formats:")
print("  .txt  → plain text files (PubMed abstracts, notes)")
print("  .pdf  → research papers, drug handbooks")
print("  .docx → clinical guidelines, protocols")
print()
print("In Colab, use the Files panel (left sidebar) to upload.")
print("Or run: from google.colab import files; files.upload()")
print()
print("The BC5CDR PubTator files (if uploaded to /content/) will also")
print("be automatically added to the RAG corpus.")

In [ ]:
# ─── Document reader functions ─────────────────────────────────────────────

def read_txt(filepath):
    """Read a plain text file."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_pdf(filepath):
    """Extract text from a PDF file."""
    text = []
    reader = pypdf.PdfReader(filepath)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

def read_docx(filepath):
    """Extract text from a .docx file."""
    doc = python_docx.Document(filepath)
    return "\n".join([para.text for para in doc.paragraphs if para.text.strip()])

def load_document(filepath):
    """Load any supported document type."""
    ext = Path(filepath).suffix.lower()
    if ext == ".txt":
        return read_txt(filepath)
    elif ext == ".pdf":
        return read_pdf(filepath)
    elif ext == ".docx":
        return read_docx(filepath)
    else:
        print(f"  [SKIP] Unsupported format: {filepath}")
        return None

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split text into overlapping chunks.
    Tries to break at sentence boundaries for cleaner chunks.
    """
    # Split into sentences first
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) <= chunk_size:
            current += " " + sent
        else:
            if current.strip():
                chunks.append(current.strip())
            # Start new chunk with overlap from previous
            overlap_text = current[-overlap:] if len(current) > overlap else current
            current = overlap_text + " " + sent
    if current.strip():
        chunks.append(current.strip())
    return [c for c in chunks if len(c) > 50]  # filter very short chunks

def load_all_documents(rag_docs_dir, bc5cdr_dir=None):
    """
    Load all documents from rag_docs_dir.
    Optionally also indexes BC5CDR PubTator files from bc5cdr_dir.
    Returns list of (doc_id, chunk_text, metadata) tuples.
    """
    all_chunks = []
    doc_count  = 0

    # Load user-uploaded documents
    supported = [".txt", ".pdf", ".docx"]
    for fpath in Path(rag_docs_dir).iterdir():
        if fpath.suffix.lower() not in supported:
            continue
        print(f"  Loading: {fpath.name} ...", end=" ")
        text = load_document(str(fpath))
        if not text:
            continue
        chunks = chunk_text(text)
        for i, chunk in enumerate(chunks):
            all_chunks.append((
                f"doc_{doc_count}_{i}",
                chunk,
                {"source": fpath.name, "chunk_idx": i, "type": "uploaded"}
            ))
        print(f"{len(chunks)} chunks")
        doc_count += 1

    # Optionally index BC5CDR training abstracts
    if bc5cdr_dir:
        pubtator_files = [
            "CDR_TrainingSet.PubTator.txt",
            "CDR_DevelopmentSet.PubTator.txt"
        ]
        for fname in pubtator_files:
            fpath = Path(bc5cdr_dir) / fname
            if not fpath.exists():
                continue
            print(f"  Loading BC5CDR: {fname} ...", end=" ")
            # Parse PubTator to extract abstracts only
            with open(str(fpath), "r", encoding="utf-8") as f:
                content = f.read()
            abstract_count = 0
            for block in content.split("\n\n"):
                lines = block.strip().split("\n")
                abstract_text = ""
                pmid = ""
                for line in lines:
                    parts = line.split("|")
                    if len(parts) >= 3 and parts[1] == "t":
                        pmid = parts[0]
                        abstract_text += parts[2] + " "
                    elif len(parts) >= 3 and parts[1] == "a":
                        abstract_text += parts[2]
                if abstract_text.strip() and pmid:
                    all_chunks.append((
                        f"bc5cdr_{pmid}",
                        abstract_text.strip(),
                        {"source": fname, "pmid": pmid, "type": "bc5cdr"}
                    ))
                    abstract_count += 1
            print(f"{abstract_count} abstracts")

    print(f"\nTotal chunks ready for embedding: {len(all_chunks)}")
    return all_chunks

print("Document loader functions defined.")

In [ ]:
# ─── Build / Load ChromaDB vector store ───────────────────────────────────

print("Initialising SentenceTransformer embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding model loaded: {EMBEDDING_MODEL}")

# Persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# Get or create the collection
# If it already exists (from a previous run), it loads the cached vectors
try:
    collection = chroma_client.get_collection(CHROMA_COLLECTION)
    print(f"Loaded existing ChromaDB collection '{CHROMA_COLLECTION}' "
          f"({collection.count()} documents)")
    REBUILD_INDEX = False
except Exception:
    collection = chroma_client.create_collection(
        CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"}  # cosine similarity
    )
    print(f"Created new ChromaDB collection '{CHROMA_COLLECTION}'")
    REBUILD_INDEX = True

def build_vector_store(force_rebuild=False):
    """
    Load all documents, embed them, and insert into ChromaDB.
    Set force_rebuild=True to re-index even if the collection exists.
    """
    global collection

    if not force_rebuild and collection.count() > 0:
        print(f"Vector store already has {collection.count()} documents. Skipping rebuild.")
        print("Pass force_rebuild=True to re-index.")
        return

    if force_rebuild:
        chroma_client.delete_collection(CHROMA_COLLECTION)
        collection = chroma_client.create_collection(
            CHROMA_COLLECTION,
            metadata={"hnsw:space": "cosine"}
        )
        print("Collection cleared for rebuild.")

    print("\nLoading documents...")
    all_chunks = load_all_documents(RAG_DOCS_DIR, bc5cdr_dir=BASE_PATH)

    if not all_chunks:
        print("WARNING: No documents found to index!")
        print(f"  → Upload documents to {RAG_DOCS_DIR}")
        print(f"  → Or upload BC5CDR PubTator files to {BASE_PATH}")
        return

    # Batch embed and insert (ChromaDB handles batching internally)
    print("\nEmbedding and indexing...")
    BATCH = 64
    ids, texts, metadatas = zip(*all_chunks)
    ids, texts, metadatas = list(ids), list(texts), list(metadatas)

    for start in range(0, len(texts), BATCH):
        end         = min(start + BATCH, len(texts))
        batch_texts = texts[start:end]
        batch_ids   = ids[start:end]
        batch_meta  = metadatas[start:end]

        embeddings  = embedder.encode(batch_texts, show_progress_bar=False).tolist()

        collection.add(
            ids        = batch_ids,
            documents  = batch_texts,
            embeddings = embeddings,
            metadatas  = batch_meta
        )
        if (start // BATCH) % 5 == 0:
            print(f"  Indexed {end}/{len(texts)} chunks...")

    print(f"\nVector store built. Total documents: {collection.count()}")

# Build the store (skips if already built)
build_vector_store(force_rebuild=REBUILD_INDEX)

In [ ]:
# ─── RAG Retriever ────────────────────────────────────────────────────────

def retrieve_evidence(query, top_k=TOP_K_RETRIEVAL, filter_type=None):
    """
    Retrieve the most relevant document chunks for a query.

    Args:
        query       : Natural language query string
        top_k       : Number of chunks to retrieve
        filter_type : Optional — 'bc5cdr' or 'uploaded' to filter by source type

    Returns:
        List of dicts with keys: text, source, score
    """
    if collection.count() == 0:
        return [{"text": "No documents indexed yet.",
                 "source": "none", "score": 0.0}]

    query_emb = embedder.encode(query).tolist()
    where     = {"type": filter_type} if filter_type else None

    results = collection.query(
        query_embeddings = [query_emb],
        n_results        = min(top_k, collection.count()),
        where            = where
    )

    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]   # cosine distance (lower = more similar)

    return [
        {
            "text"   : doc,
            "source" : meta.get("source", "unknown"),
            "score"  : round(1 - dist, 4)   # convert distance to similarity
        }
        for doc, meta, dist in zip(docs, metas, distances)
    ]

# Quick test
test_results = retrieve_evidence("cisplatin nephrotoxicity mechanism")
print("Sample RAG retrieval for 'cisplatin nephrotoxicity mechanism':")
for i, r in enumerate(test_results):
    print(f"\n  [{i+1}] Source: {r['source']}  Similarity: {r['score']}")
    print(f"       {r['text'][:200]}...")

## 4. LLM Integration

Three LLM-powered functions:
1. **`explain_relation`** — generates a clinical explanation for a predicted drug–disease pair using RAG-retrieved evidence
2. **`verify_ner`** — cross-checks BioBERT NER predictions against LLM knowledge
3. **`answer_biomedical_question`** — open-ended QA over the RAG corpus

In [ ]:
# ─── LLM call wrapper ─────────────────────────────────────────────────────

def call_llm(prompt, system_prompt=None, max_tokens=600):
    try:
        if LLM_PROVIDER == "anthropic":
            messages = [{"role": "user", "content": prompt}]
            kwargs   = {
                "model"      : ANTHROPIC_MODEL,
                "max_tokens" : max_tokens,
                "messages"   : messages
            }
            if system_prompt:
                kwargs["system"] = system_prompt
            resp = llm_client.messages.create(**kwargs)
            return resp.content[0].text

        elif LLM_PROVIDER == "openai":
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": prompt})
            resp = llm_client.chat.completions.create(
                model      = OPENAI_MODEL,
                max_tokens = max_tokens,
                messages   = messages
            )
            return resp.choices[0].message.content

        else:  # gemini
            full_prompt = f"{system_prompt}\n\n{prompt}" if system_prompt else prompt
            resp = llm_client.generate_content(full_prompt)
            return resp.text

    except Exception as e:
        return f"[LLM ERROR: {str(e)}]"

## 5. Data Preprocessing (BC5CDR)

In [ ]:
# ─── PubTator parser ──────────────────────────────────────────────────────
def parse_pubtator(filepath):
    documents = []
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    blocks = [b.strip() for b in content.strip().split("\n\n") if b.strip()]
    for block in blocks:
        lines = block.split("\n")
        doc   = {"pmid": None, "title": "", "abstract": "",
                 "entities": [], "relations": []}
        for line in lines:
            parts = line.split("|")
            if len(parts) >= 3 and parts[1] == "t":
                doc["pmid"]    = parts[0]
                doc["title"]   = parts[2]
            elif len(parts) >= 3 and parts[1] == "a":
                doc["abstract"] = parts[2]
            else:
                cols = line.split("\t")
                if len(cols) == 6:
                    doc["entities"].append({
                        "pmid":    cols[0],
                        "start":   int(cols[1]),
                        "end":     int(cols[2]),
                        "text":    cols[3],
                        "type":    cols[4],
                        "mesh_id": cols[5]
                    })
                elif len(cols) == 4 and cols[1] == "CID":
                    doc["relations"].append({
                        "pmid":     cols[0],
                        "rel_type": cols[1],
                        "chemical": cols[2],
                        "disease":  cols[3]
                    })
        if doc["pmid"]:
            documents.append(doc)
    return documents

train_docs = parse_pubtator(BASE_PATH + "CDR_TrainingSet.PubTator.txt")
val_docs   = parse_pubtator(BASE_PATH + "CDR_DevelopmentSet.PubTator.txt")
test_docs  = parse_pubtator(BASE_PATH + "CDR_TestSet.PubTator.txt")

print(f"Loaded → train:{len(train_docs)}  val:{len(val_docs)}  test:{len(test_docs)}")
print(f"\nSample document:")
print(f"  PMID     : {train_docs[0]['pmid']}")
print(f"  Title    : {train_docs[0]['title'][:80]}")
print(f"  Entities : {train_docs[0]['entities'][:2]}")
print(f"  Relations: {train_docs[0]['relations'][:2]}")

In [ ]:
NER_LABELS = ["O", "B-Chemical", "I-Chemical", "B-Disease", "I-Disease"]
LABEL2ID   = {l: i for i, l in enumerate(NER_LABELS)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def doc_to_ner_row(doc):
    full_text = doc["title"] + " " + doc["abstract"]
    words     = full_text.split()
    tags      = ["O"] * len(words)
    char2word = {}
    cursor    = 0
    for wi, word in enumerate(words):
        for ci in range(len(word)):
            char2word[cursor + ci] = wi
        cursor += len(word) + 1
    for ent in doc["entities"]:
        etype = ent["type"]
        s, e  = ent["start"], ent["end"]
        span_word_indices = sorted(set(
            char2word[c] for c in range(s, e) if c in char2word
        ))
        for i, wi in enumerate(span_word_indices):
            prefix   = "B-" if i == 0 else "I-"
            tags[wi] = f"{prefix}{etype}"
    return {"tokens": words, "ner_tags": tags}

def doc_to_re_rows(doc):
    rows      = []
    text      = doc["title"] + " " + doc["abstract"]
    chemicals = [e for e in doc["entities"] if e["type"] == "Chemical"]
    diseases  = [e for e in doc["entities"] if e["type"] == "Disease"]
    cid_pairs = set(
        (r["chemical"].strip(), r["disease"].strip())
        for r in doc["relations"]
    )
    for chem in chemicals:
        for dis in diseases:
            chem_mesh = chem["mesh_id"].strip()
            dis_mesh  = dis["mesh_id"].strip()
            label     = 1 if (chem_mesh, dis_mesh) in cid_pairs else 0
            pair_text = (
                f"[CHEM] {chem['text']} [/CHEM] "
                f"[DIS] {dis['text']} [/DIS] "
                f"[TEXT] {text[:300]}"
            )
            rows.append({"text": pair_text, "label": label})
    return rows

def docs_to_re_dataset(docs):
    all_rows = []
    for doc in docs:
        all_rows.extend(doc_to_re_rows(doc))
    return all_rows

train_ner = [doc_to_ner_row(d) for d in train_docs]
val_ner   = [doc_to_ner_row(d) for d in val_docs]
test_ner  = [doc_to_ner_row(d) for d in test_docs]

train_re  = docs_to_re_dataset(train_docs)
val_re    = docs_to_re_dataset(val_docs)
test_re   = docs_to_re_dataset(test_docs)

print("NER Label Distribution:")
train_tag_counts = Counter(t for r in train_ner for t in r["ner_tags"])
for tag in NER_LABELS:
    print(f"  {tag:<20} {train_tag_counts[tag]}")

print("\nRE Label Distribution:")
for split_name, split in [("Train", train_re), ("Val", val_re), ("Test", test_re)]:
    counts = Counter(r["label"] for r in split)
    total  = len(split)
    print(f"  {split_name} → total:{total}  "
          f"CID:{counts[1]} ({100*counts[1]/total:.1f}%)  "
          f"No CID:{counts[0]} ({100*counts[0]/total:.1f}%)")

In [ ]:
def align_labels_with_tokens(tokens, ner_tags, tokenizer):
    enc      = tokenizer(tokens, is_split_into_words=True,
                         truncation=True, max_length=512)
    word_ids = enc.word_ids()
    aligned  = []
    prev_wid = None
    for wid in word_ids:
        if wid is None:
            aligned.append(-100)
        elif wid != prev_wid:
            aligned.append(LABEL2ID.get(ner_tags[wid], 0))
        else:
            tag = ner_tags[wid]
            aligned.append(LABEL2ID.get(tag.replace("B-", "I-"), 0))
        prev_wid = wid
    enc["labels"] = aligned
    return enc

class NERDataset(Dataset):
    def __init__(self, rows):
        self.data = [
            align_labels_with_tokens(r["tokens"], r["ner_tags"], tokenizer)
            for r in rows
        ]
    def __len__(self):        return len(self.data)
    def __getitem__(self, i): return self.data[i]

class REDataset(Dataset):
    def __init__(self, rows, max_len=256):
        self.encodings = tokenizer(
            [r["text"] for r in rows],
            truncation=True, padding=True, max_length=max_len
        )
        self.labels = [r["label"] for r in rows]
    def __len__(self):        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

class JointDataset(Dataset):
    def __init__(self, ner_rows, re_rows):
        assert len(ner_rows) == len(re_rows)
        self.ner_encs  = [
            align_labels_with_tokens(r["tokens"], r["ner_tags"], tokenizer)
            for r in ner_rows
        ]
        self.re_labels = [r["label"] for r in re_rows]
    def __len__(self): return len(self.re_labels)
    def __getitem__(self, i):
        enc = self.ner_encs[i]
        return {
            "input_ids":      torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "ner_labels":     torch.tensor(enc["labels"]),
            "re_labels":      torch.tensor(self.re_labels[i])
        }

train_ner_ds = NERDataset(train_ner)
val_ner_ds   = NERDataset(val_ner)
test_ner_ds  = NERDataset(test_ner)

train_re_ds  = REDataset(train_re)
val_re_ds    = REDataset(val_re)
test_re_ds   = REDataset(test_re)

min_train      = min(len(train_ner), len(train_re))
min_val        = min(len(val_ner),   len(val_re))
min_test       = min(len(test_ner),  len(test_re))
joint_train_ds = JointDataset(train_ner[:min_train], train_re[:min_train])
joint_val_ds   = JointDataset(val_ner[:min_val],     val_re[:min_val])
joint_test_ds  = JointDataset(test_ner[:min_test],   test_re[:min_test])

print("All datasets ready.")
print(f"  NER   — train:{len(train_ner_ds)} val:{len(val_ner_ds)} test:{len(test_ner_ds)}")
print(f"  RE    — train:{len(train_re_ds)}  val:{len(val_re_ds)}  test:{len(test_re_ds)}")
print(f"  Joint — train:{len(joint_train_ds)} val:{len(joint_val_ds)} test:{len(joint_test_ds)}")

## 6. Training — Pipeline Model (BioBERT NER + RE)

In [ ]:
def compute_ner_label_weights(ner_rows, label2id):
    counts  = Counter(label2id.get(tag, 0) for row in ner_rows for tag in row["ner_tags"])
    total   = sum(counts.values())
    n_cls   = len(label2id)
    weights = [total / (n_cls * counts.get(i, 1)) for i in range(n_cls)]
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)

ner_weights = compute_ner_label_weights(train_ner, LABEL2ID)
print("NER class weights:")
for label, lid in LABEL2ID.items():
    print(f"  {label:<20} {ner_weights[lid]:.4f}")

class WeightedNERTrainer(Trainer):
    def __init__(self, label_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_weights = label_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=self.label_weights, ignore_index=-100)
        loss    = loss_fn(
            outputs.logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

def compute_ner_metrics(p):
    preds  = np.argmax(p.predictions, axis=2)
    labels = p.label_ids
    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tp, tl = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                tp.append(ID2LABEL[p_id])
                tl.append(ID2LABEL[l_id])
        true_preds.append(tp)
        true_labels.append(tl)
    return {"f1": seqeval_f1(true_labels, true_preds)}

collator  = DataCollatorForTokenClassification(tokenizer)
ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(NER_LABELS),
    id2label=ID2LABEL, label2id=LABEL2ID
)

ner_args = TrainingArguments(
    output_dir                  = "./ner_model",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    learning_rate               = 3e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    logging_steps               = 20,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none"
)

ner_trainer = WeightedNERTrainer(
    label_weights = ner_weights,
    model         = ner_model,
    args          = ner_args,
    train_dataset = train_ner_ds,
    eval_dataset  = val_ner_ds,
    data_collator = collator,
    compute_metrics = compute_ner_metrics,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=3)]
)
print("Training NER model...")
ner_trainer.train()
print("NER training complete.")

In [ ]:
re_counts        = Counter(r["label"] for r in train_re)
n_neg, n_pos     = re_counts[0], re_counts[1]
re_weights       = torch.tensor([1.0/n_neg, 1.0/n_pos], dtype=torch.float)
re_weights       = (re_weights / re_weights.sum()).to(DEVICE)
print(f"RE class weights → No CID:{re_weights[0]:.4f}  CID:{re_weights[1]:.4f}")

class WeightedRETrainer(Trainer):
    def __init__(self, label_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_weights = label_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=self.label_weights)
        loss    = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_re_metrics(p):
    preds  = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    return {"f1": f1_score(labels, preds, average="binary")}

re_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

re_args = TrainingArguments(
    output_dir                  = "./re_model",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    learning_rate               = 3e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    logging_steps               = 20,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none"
)

re_trainer = WeightedRETrainer(
    label_weights = re_weights,
    model         = re_model,
    args          = re_args,
    train_dataset = train_re_ds,
    eval_dataset  = val_re_ds,
    compute_metrics = compute_re_metrics,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=3)]
)
print("Training RE model...")
re_trainer.train()
print("RE training complete.")

## 7. Training — Joint Model (Multi-Task BioBERT)

In [ ]:
class JointNERREModel(nn.Module):
    def __init__(self, model_name, num_ner_labels, num_re_labels=2):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        hidden        = self.encoder.config.hidden_size
        self.ner_head = nn.Linear(hidden, num_ner_labels)
        self.re_head  = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(256, num_re_labels)
        )
        self.num_ner  = num_ner_labels

    def forward(self, input_ids, attention_mask,
                token_type_ids=None, ner_labels=None, re_labels=None):
        out        = self.encoder(
                         input_ids=input_ids,
                         attention_mask=attention_mask,
                         token_type_ids=token_type_ids
                     )
        seq_out    = out.last_hidden_state
        cls_out    = seq_out[:, 0, :]
        ner_logits = self.ner_head(seq_out)
        re_logits  = self.re_head(cls_out)
        loss       = None
        if ner_labels is not None and re_labels is not None:
            ce       = nn.CrossEntropyLoss(ignore_index=-100)
            ner_loss = ce(ner_logits.view(-1, self.num_ner), ner_labels.view(-1))
            re_loss  = ce(re_logits, re_labels)
            loss     = ner_loss + re_loss
        return {"loss": loss, "ner_logits": ner_logits, "re_logits": re_logits}

joint_model = JointNERREModel(MODEL_NAME, num_ner_labels=len(NER_LABELS)).to(DEVICE)
print(f"Joint model parameters: {sum(p.numel() for p in joint_model.parameters()):,}")

In [ ]:
EPOCHS = 5
BATCH  = 8

def joint_collate_fn(batch):
    input_ids      = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    ner_labels     = [item["ner_labels"] for item in batch]
    re_labels      = torch.tensor([item["re_labels"] for item in batch], dtype=torch.long)
    max_len        = max(len(x) for x in input_ids)
    padded_input_ids = torch.stack([
        torch.cat([x, torch.tensor([tokenizer.pad_token_id] * (max_len - len(x)), dtype=torch.long)])
        for x in input_ids
    ])
    padded_attention_mask = torch.stack([
        torch.cat([x, torch.tensor([0] * (max_len - len(x)), dtype=torch.long)])
        for x in attention_mask
    ])
    padded_ner_labels = torch.stack([
        torch.cat([x, torch.tensor([-100] * (max_len - len(x)), dtype=torch.long)])
        for x in ner_labels
    ])
    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "ner_labels": padded_ner_labels,
        "re_labels": re_labels
    }

train_loader = DataLoader(joint_train_ds, batch_size=BATCH, shuffle=True,  collate_fn=joint_collate_fn)
val_loader   = DataLoader(joint_val_ds,   batch_size=BATCH, collate_fn=joint_collate_fn)

optimizer    = AdamW(joint_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(
                   optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
               )

best_val_loss = float("inf")
patience, wait = 3, 0

print("Training Joint model...")
for epoch in range(EPOCHS):
    joint_model.train()
    total_train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = joint_model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            ner_labels=batch["ner_labels"].to(DEVICE),
            re_labels=batch["re_labels"].to(DEVICE)
        )
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(joint_model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_train_loss += out["loss"].item()

    joint_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            out = joint_model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                ner_labels=batch["ner_labels"].to(DEVICE),
                re_labels=batch["re_labels"].to(DEVICE)
            )
            total_val_loss += out["loss"].item()

    avg_train = total_train_loss / len(train_loader)
    avg_val   = total_val_loss   / len(val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — train:{avg_train:.4f}  val:{avg_val:.4f}", end="")

    if avg_val < best_val_loss:
        best_val_loss = avg_val; wait = 0
        torch.save(joint_model.state_dict(), "best_joint_model.pt")
        print("  ✓ saved")
    else:
        wait += 1
        print(f"  (no improvement {wait}/{patience})")
        if wait >= patience:
            print("Early stopping triggered."); break

joint_model.load_state_dict(torch.load("best_joint_model.pt", weights_only=True))
print("Best joint model restored.")

## 8. Evaluation

In [ ]:
# ─── Pipeline NER evaluation ──────────────────────────────────────────────
ner_preds_out = ner_trainer.predict(test_ner_ds)
ner_pred_ids  = np.argmax(ner_preds_out.predictions, axis=2)
ner_true_ids  = ner_preds_out.label_ids

true_tags, pred_tags = [], []
for pred_seq, label_seq in zip(ner_pred_ids, ner_true_ids):
    tp, tl = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id != -100:
            tp.append(ID2LABEL[p_id])
            tl.append(ID2LABEL[l_id])
    true_tags.append(tl); pred_tags.append(tp)

print("=" * 55)
print("Pipeline A — NER Report")
print("=" * 55)
print(ner_report(true_tags, pred_tags))

# ─── Pipeline RE evaluation ───────────────────────────────────────────────
re_preds_out = re_trainer.predict(test_re_ds)
re_pred_ids  = np.argmax(re_preds_out.predictions, axis=1)
re_true_ids  = re_preds_out.label_ids

print("=" * 55)
print("Pipeline A — RE Report")
print("=" * 55)
print(classification_report(re_true_ids, re_pred_ids, target_names=["No CID", "CID"]))

In [ ]:
test_loader = DataLoader(joint_test_ds, batch_size=16, collate_fn=joint_collate_fn)

joint_model.eval()
all_ner_preds, all_ner_true = [], []
all_re_preds,  all_re_true  = [], []

with torch.no_grad():
    for batch in test_loader:
        out   = joint_model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE)
        )
        ner_p = torch.argmax(out["ner_logits"], dim=2).cpu().numpy()
        ner_l = batch["ner_labels"].numpy()
        for ps, ls in zip(ner_p, ner_l):
            tp, tl = [], []
            for p, l in zip(ps, ls):
                if l != -100:
                    tp.append(ID2LABEL[p]); tl.append(ID2LABEL[l])
            all_ner_preds.append(tp); all_ner_true.append(tl)
        all_re_preds.extend(torch.argmax(out["re_logits"], 1).cpu().numpy())
        all_re_true.extend(batch["re_labels"].numpy())

print("=" * 55)
print("Joint Model — NER Report")
print("=" * 55)
print(ner_report(all_ner_true, all_ner_preds))

print("=" * 55)
print("Joint Model — RE Report")
print("=" * 55)
print(classification_report(all_re_true, all_re_preds, target_names=["No CID", "CID"]))

In [ ]:
# ─── Accuracy summary ─────────────────────────────────────────────────────
pipe_ner_flat_true  = [t for seq in true_tags     for t in seq]
pipe_ner_flat_pred  = [t for seq in pred_tags     for t in seq]
joint_ner_flat_true = [t for seq in all_ner_true  for t in seq]
joint_ner_flat_pred = [t for seq in all_ner_preds for t in seq]

acc_df = pd.DataFrame({
    "Model"       : ["Pipeline", "Joint"],
    "NER Accuracy": [round(accuracy_score(pipe_ner_flat_true, pipe_ner_flat_pred), 4),
                     round(accuracy_score(joint_ner_flat_true, joint_ner_flat_pred), 4)],
    "RE Accuracy" : [round(accuracy_score(re_true_ids, re_pred_ids), 4),
                     round(accuracy_score(all_re_true, all_re_preds), 4)]
})
print("=" * 55)
print("ACCURACY SUMMARY")
print("=" * 55)
print(acc_df.to_string(index=False))

## 9. Explainability — SHAP, LIME, Attention

In [ ]:
sys.setrecursionlimit(5000)

re_pipe = hf_pipeline(
    "text-classification", model=re_model, tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True
)
explainer_pipe   = shap.Explainer(re_pipe)
sample_texts     = [r["text"] for r in test_re[:10]]
shap_values_pipe = explainer_pipe(sample_texts)

print("SHAP — Pipeline A (first sample):")
shap.plots.text(shap_values_pipe[0])

class JointREWrapper:
    def __call__(self, texts):
        joint_model.eval()
        probabilities = []
        for text in texts:
            enc = tokenizer(text, return_tensors="pt",
                            truncation=True, padding=True,
                            max_length=256).to(DEVICE)
            with torch.no_grad():
                out  = joint_model(input_ids=enc["input_ids"],
                                   attention_mask=enc["attention_mask"])
                prob = torch.softmax(out["re_logits"], dim=1).cpu().numpy()[0].astype(np.float64)
            probabilities.append(prob)
        return np.array(probabilities, dtype=np.float64)

joint_masker      = shap.maskers.Text(tokenizer, mask_token=tokenizer.mask_token)
explainer_joint   = shap.Explainer(JointREWrapper(), joint_masker)
shap_values_joint = explainer_joint(sample_texts)

print("SHAP — Joint Model (first sample):")
shap.plots.text(shap_values_joint[0])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, sv, title in [
    (axes[0], shap_values_pipe,  "Pipeline A — SHAP Top Features"),
    (axes[1], shap_values_joint, "Joint Model — SHAP Top Features")
]:
    vals     = sv.values[..., 1] if sv.values.ndim == 3 else sv.values
    names    = sv[0].feature_names
    mean_abs = np.abs(vals).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-min(15, len(names)):]
    ax.barh([names[i] for i in top_idx], [mean_abs[i] for i in top_idx], color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("Mean |SHAP value|")
plt.suptitle("SHAP Feature Importance — Both Models", fontsize=13)
plt.tight_layout()
plt.savefig("shap_comparison.png", dpi=150)
plt.show()

In [ ]:
lime_exp = LimeTextExplainer(class_names=["No CID", "CID"])

def pipe_predict_proba(texts):
    probabilities = []
    for text in texts:
        prediction_list = re_pipe(text)
        # HF pipeline returns list-of-lists when return_all_scores=True
        scores = prediction_list[0] if isinstance(prediction_list[0], list) else prediction_list
        scores_map = {item['label']: item['score'] for item in scores}
        prob_no_cid = scores_map.get('LABEL_0', scores_map.get('No CID', 0.0))
        prob_cid    = scores_map.get('LABEL_1', scores_map.get('CID', 0.0))
        probabilities.append([prob_no_cid, prob_cid])
    return np.array(probabilities)

def joint_predict_proba(texts):
    joint_model.eval()
    probs = []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt",
                        truncation=True, padding=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out  = joint_model(input_ids=enc["input_ids"],
                               attention_mask=enc["attention_mask"])
            prob = torch.softmax(out["re_logits"], dim=1).cpu().numpy()[0]
        probs.append(prob)
    return np.array(probs)

sample_text = test_re[0]["text"]
exp_pipe    = lime_exp.explain_instance(sample_text, pipe_predict_proba,  num_features=15, num_samples=300)
exp_joint   = lime_exp.explain_instance(sample_text, joint_predict_proba, num_features=15, num_samples=300)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, exp, title in [
    (axes[0], exp_pipe,  "Pipeline A — LIME"),
    (axes[1], exp_joint, "Joint Model — LIME")
]:
    feats   = exp.as_list()
    words   = [f[0] for f in feats]
    weights = [f[1] for f in feats]
    colors  = ["#4C72B0" if w > 0 else "#DD8452" for w in weights]
    ax.barh(words, weights, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("LIME Weight")
plt.suptitle("LIME Explanation — Both Models", fontsize=13)
plt.tight_layout()
plt.savefig("lime_comparison.png", dpi=150)
plt.show()

In [ ]:
def get_ner_attention(text, model, model_type="pipeline"):
    words = text.split()[:40]
    enc   = tokenizer(words, is_split_into_words=True,
                      return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    word_ids = tokenizer(words, is_split_into_words=True,
                         truncation=True, max_length=256).word_ids()
    with torch.no_grad():
        if model_type == "pipeline":
            if hasattr(model.config, '_attn_implementation') and model.config._attn_implementation == 'sdpa':
                model.config._attn_implementation = 'eager'
            model.config.output_attentions = True
            out        = model(**enc, output_attentions=True)
            logits     = out.logits
            attentions = out.attentions
        else:
            if hasattr(model.encoder.config, '_attn_implementation') and model.encoder.config._attn_implementation == 'sdpa':
                model.encoder.config._attn_implementation = 'eager'
            model.encoder.config.output_attentions = True
            enc_out    = model.encoder(**enc, output_attentions=True)
            logits     = model.ner_head(enc_out.last_hidden_state)
            attentions = enc_out.attentions

    cls_attn = attentions[-1][0].mean(dim=0)[0].cpu().numpy()
    pred_ids = torch.argmax(logits, dim=2)[0].cpu().numpy()
    tokens   = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    result_words, result_preds, result_attns = [], [], []
    seen = set()
    for idx, (tok, wid) in enumerate(zip(tokens, word_ids)):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        result_words.append(words[wid])
        result_preds.append(ID2LABEL[pred_ids[idx]])
        result_attns.append(float(cls_attn[idx]))
    return result_words, result_preds, result_attns

def plot_ner_attention(words, preds, attns, title, ax):
    norm = mcolors.Normalize(vmin=min(attns), vmax=max(attns))
    cmap = plt.cm.YlOrRd
    ax.set_xlim(0, len(words))
    ax.set_ylim(0, 1.6)
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    for i, (word, pred, attn) in enumerate(zip(words, preds, attns)):
        color = cmap(norm(attn))
        rect  = plt.Rectangle((i, 1.0), 0.9, 0.4, facecolor=color, edgecolor="gray", linewidth=0.5)
        ax.add_patch(rect)
        ax.text(i+0.45, 1.2, word, ha="center", va="center", fontsize=7, color="black")
        tag_color = ("#4C72B0" if "Chemical" in pred else "#DD8452" if "Disease" in pred else "#aaaaaa")
        ax.text(i+0.45, 0.65, pred.replace("B-","").replace("I-",""),
                ha="center", va="center", fontsize=6, color=tag_color, fontweight="bold")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.02, label="Attention")

sample_text = test_docs[0]["title"] + " " + test_docs[0]["abstract"]
words_p, preds_p, attns_p = get_ner_attention(sample_text, ner_model,   model_type="pipeline")
words_j, preds_j, attns_j = get_ner_attention(sample_text, joint_model, model_type="joint")

fig, axes = plt.subplots(2, 1, figsize=(18, 7))
plot_ner_attention(words_p, preds_p, attns_p, "Pipeline A — NER Token Attention",  axes[0])
plot_ner_attention(words_j, preds_j, attns_j, "Joint Model — NER Token Attention", axes[1])
plt.suptitle("NER Token-level Explainability", fontsize=13)
plt.tight_layout()
plt.savefig("ner_attention.png", dpi=150)
plt.show()

## 10. LLM + RAG Enhanced Pipeline

This section wires together BioBERT predictions → RAG retrieval → LLM explanation into a complete end-to-end system.

In [ ]:
# ─── Full enhanced extraction pipeline ────────────────────────────────────

def extract_drug_disease_pairs_enhanced(text, threshold=0.5, use_rag=True,
                                         use_llm=True, verify_ner=False):
    """
    Full pipeline:
      1. BioBERT NER  → extract Chemical + Disease entities
      2. Optional: LLM NER verification
      3. BioBERT RE   → score all (Chemical × Disease) pairs
      4. Optional: RAG retrieval of supporting evidence
      5. Optional: LLM clinical explanation

    Args:
        text       : Input biomedical text
        threshold  : Confidence threshold for reporting CID pairs
        use_rag    : Whether to retrieve supporting evidence from ChromaDB
        use_llm    : Whether to generate LLM clinical explanations
        verify_ner : Whether to run LLM NER verification step

    Returns:
        List of result dicts
    """
    print("─" * 60)
    print("STEP 1: BioBERT Named Entity Recognition")
    print("─" * 60)

    # ── NER ───────────────────────────────────────────────────────────────
    ner_pipe = hf_pipeline(
        "ner", model=ner_model, tokenizer=tokenizer,
        aggregation_strategy="simple",
        device=0 if torch.cuda.is_available() else -1
    )
    ner_out   = ner_pipe(text)
    chemicals = [e["word"] for e in ner_out if "Chemical" in e["entity_group"]]
    diseases  = [e["word"] for e in ner_out if "Disease"  in e["entity_group"]]

    print(f"  Chemicals detected : {chemicals}")
    print(f"  Diseases detected  : {diseases}")

    # ── Optional: LLM NER verification ────────────────────────────────────
    if verify_ner and use_llm:
        print("\nSTEP 1b: LLM NER Verification")
        print("─" * 60)
        bert_preds = [(e["word"], e["entity_group"]) for e in ner_out]
        verification = verify_ner_with_llm(text, bert_preds)
        print(f"  LLM Verification:\n  {verification}")

    if not chemicals or not diseases:
        print("  No drug-disease pairs to evaluate.")
        return []

    print(f"\nSTEP 2: BioBERT Relation Extraction")
    print("─" * 60)
    print(f"  Evaluating {len(chemicals) * len(diseases)} pair(s)...")

    # ── RE ────────────────────────────────────────────────────────────────
    re_model.eval()
    results = []
    for chem in chemicals:
        for dis in diseases:
            probe = (
                f"[CHEM] {chem} [/CHEM] "
                f"[DIS] {dis} [/DIS] "
                f"[TEXT] {text[:300]}"
            )
            enc  = tokenizer(probe, return_tensors="pt",
                             truncation=True, max_length=256).to(DEVICE)
            with torch.no_grad():
                logits = re_model(**enc).logits
            prob = torch.softmax(logits, dim=1)[0][1].item()

            if prob < threshold:
                continue

            print(f"  ✓ {chem} → {dis}  (confidence: {prob:.3f})")

            result = {
                "drug"          : chem,
                "disease"       : dis,
                "confidence"    : round(prob, 4),
                "rag_evidence"  : [],
                "llm_explanation": ""
            }

            # ── RAG evidence retrieval ─────────────────────────────────────
            if use_rag:
                print(f"\nSTEP 3: RAG Evidence Retrieval for ({chem} → {dis})")
                print("─" * 60)
                evidence = retrieve_evidence(
                    f"{chem} {dis} adverse effect mechanism",
                    top_k=TOP_K_RETRIEVAL
                )
                result["rag_evidence"] = evidence
                for i, ev in enumerate(evidence):
                    print(f"  [{i+1}] {ev['source']} (sim={ev['score']})")
                    print(f"       {ev['text'][:150]}...")

            # ── LLM explanation ────────────────────────────────────────────
            if use_llm:
                print(f"\nSTEP 4: LLM Clinical Explanation")
                print("─" * 60)
                explanation, _ = explain_relation(
                    drug        = chem,
                    disease     = dis,
                    source_text = text,
                    confidence  = prob,
                    use_rag     = use_rag
                )
                result["llm_explanation"] = explanation
                print(f"  {explanation}")

            results.append(result)

    return results

print("Enhanced pipeline function defined.")

In [ ]:
import requests

HF_TOKEN = ""  # paste your token here
API_URL = "https://api-inference.huggingface.co/models/meta-llama/Meta-Llama-3-8B-Instruct"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

payload = {
    "inputs": "<|begin_of_text|><|user|>\nSay: hello\n<|assistant|>\n",
    "parameters": {"max_new_tokens": 20}
}

resp = requests.post(API_URL, headers=headers, json=payload)
print(resp.json())

In [ ]:
# ─── Demo 1: Synthetic biomedical text ────────────────────────────────────

demo_text = (
    "Administration of cisplatin has been associated with nephrotoxicity "
    "and peripheral neuropathy in cancer patients. Carboplatin shows "
    "reduced nephrotoxicity compared to cisplatin but may cause "
    "thrombocytopenia. Both agents have been linked to ototoxicity "
    "following high-dose chemotherapy regimens."
)

print("=" * 60)
print("DEMO 1: SYNTHETIC BIOMEDICAL TEXT")
print("=" * 60)
print(f"Input: {demo_text}\n")

demo_results = extract_drug_disease_pairs_enhanced(
    text       = demo_text,
    threshold  = 0.4,
    use_rag    = True,
    use_llm    = True,
    verify_ner = True
)

print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
if demo_results:
    for r in demo_results:
        print(f"\n  Drug    : {r['drug']}")
        print(f"  Disease : {r['disease']}")
        print(f"  Score   : {r['confidence']}")
        print(f"  Evidence: {len(r['rag_evidence'])} chunk(s) retrieved")
        if r['llm_explanation']:
            print(f"  LLM     : {r['llm_explanation'][:200]}...")
else:
    print("  No pairs above threshold.")

In [ ]:
# ─── Demo 2: Real BC5CDR test documents ───────────────────────────────────

print("=" * 60)
print("DEMO 2: REAL BC5CDR TEST DOCUMENTS")
print("=" * 60)

for doc in test_docs[:3]:
    text      = doc["title"] + " " + doc["abstract"]
    true_rels = [(r["chemical"], r["disease"]) for r in doc["relations"]]

    print(f"\nPMID: {doc['pmid']}")
    print(f"True relations: {true_rels}")
    print()

    results = extract_drug_disease_pairs_enhanced(
        text      = text,
        threshold = 0.5,
        use_rag   = True,
        use_llm   = True
    )

    print(f"\nPredicted pairs:")
    for r in results:
        print(f"  {r['drug']:20s} → {r['disease']:25s} (conf: {r['confidence']})")
    print("-" * 60)

## 11. RAG-Powered Biomedical QA

Ask any question — the system retrieves relevant evidence from your indexed documents and the BC5CDR corpus, then uses the LLM to formulate a grounded answer.

In [ ]:
# ─── Interactive biomedical QA ─────────────────────────────────────────────

questions = [
    "What are the known adverse effects of cisplatin?",
    "Which drugs are associated with causing peripheral neuropathy?",
    "What is the mechanism by which methotrexate causes hepatotoxicity?"
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}")
    print("=" * 60)
    answer, sources = biomedical_qa(q)
    print(f"A: {answer}")
    print(f"\nSources used:")
    for s in sources:
        print(f"  - {s['source']} (similarity: {s['score']})")
    print()

In [ ]:
# ─── Ask your own question ─────────────────────────────────────────────────
# Change this string to ask anything about the biomedical corpus

YOUR_QUESTION = "Does carboplatin cause thrombocytopenia and what is the clinical management?"

print(f"Q: {YOUR_QUESTION}")
print("=" * 60)
answer, sources = biomedical_qa(YOUR_QUESTION)
print(f"A: {answer}")
print(f"\nSources: {[s['source'] for s in sources]}")

## 12. ChromaDB Management Utilities

Helper cells for managing the vector store — add new documents, inspect contents, or rebuild from scratch.

In [ ]:
# ─── Add a single document to the store on-the-fly ────────────────────────

def add_document_to_store(text, doc_id, source_name):
    """
    Add a single text document to ChromaDB without rebuilding.
    Useful for adding one abstract, clinical note, or text snippet.

    Args:
        text        : Raw text to index
        doc_id      : Unique identifier string
        source_name : Label shown in retrieval results
    """
    chunks = chunk_text(text)
    ids, docs, metas, embs = [], [], [], []
    for i, chunk in enumerate(chunks):
        chunk_id = f"{doc_id}_chunk{i}"
        ids.append(chunk_id)
        docs.append(chunk)
        metas.append({"source": source_name, "chunk_idx": i, "type": "uploaded"})
        embs.append(embedder.encode(chunk).tolist())

    collection.add(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
    print(f"Added {len(chunks)} chunk(s) from '{source_name}' to ChromaDB.")
    print(f"Total documents in store: {collection.count()}")

# ─── Inspect what's in the store ─────────────────────────────────────────
def inspect_store(n=5):
    """Print a sample of what's in the ChromaDB collection."""
    total = collection.count()
    print(f"ChromaDB collection '{CHROMA_COLLECTION}': {total} documents")
    if total == 0:
        print("  (empty)")
        return
    sample = collection.get(limit=n, include=["documents", "metadatas"])
    print(f"\nFirst {min(n, total)} entries:")
    for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"])):
        print(f"  [{i+1}] source={meta.get('source','?')} type={meta.get('type','?')}")
        print(f"       {doc[:120]}...")

# ─── Delete and rebuild ───────────────────────────────────────────────────
def rebuild_store():
    """Wipe and rebuild the ChromaDB store from scratch."""
    build_vector_store(force_rebuild=True)

# Run inspection
inspect_store()

In [ ]:
# ─── Example: Add a custom abstract or note ───────────────────────────────
# Replace the text below with any biomedical content you want to index

CUSTOM_TEXT = """
Doxorubicin (adriamycin) is an anthracycline antibiotic widely used in cancer chemotherapy.
Its principal dose-limiting toxicity is cardiomyopathy, which can progress to congestive
heart failure. The mechanism involves free radical generation and mitochondrial dysfunction
in cardiac myocytes. Cumulative dose above 400-550 mg/m2 significantly increases cardiac
risk. Liposomal formulations have been developed to reduce cardiotoxicity while preserving
antitumor activity.
"""

# Uncomment to add this document:
# add_document_to_store(
#     text        = CUSTOM_TEXT,
#     doc_id      = "doxorubicin_cardiotox",
#     source_name = "custom_note_doxorubicin.txt"
# )

## 13. Model Comparison & Error Analysis

In [ ]:
def get_ner_metrics_per_type(true_seqs, pred_seqs):
    rows = []
    for etype in ["Chemical", "Disease"]:
        ft, fp = [], []
        for ts, ps in zip(true_seqs, pred_seqs):
            ft.append([t if etype in t else "O" for t in ts])
            fp.append([p if etype in p else "O" for p in ps])
        rows.append({
            "Entity"   : etype,
            "Precision": round(seqeval_precision(ft, fp), 4),
            "Recall"   : round(seqeval_recall(ft, fp),    4),
            "F1"       : round(seqeval_f1(ft, fp),        4)
        })
    rows.append({
        "Entity"   : "Overall",
        "Precision": round(seqeval_precision(true_seqs, pred_seqs), 4),
        "Recall"   : round(seqeval_recall(true_seqs, pred_seqs),    4),
        "F1"       : round(seqeval_f1(true_seqs, pred_seqs),        4)
    })
    return pd.DataFrame(rows)

df_pipe_ner  = get_ner_metrics_per_type(true_tags,    pred_tags)
df_joint_ner = get_ner_metrics_per_type(all_ner_true, all_ner_preds)
df_pipe_ner["Model"]  = "Pipeline A"
df_joint_ner["Model"] = "Joint Model"

df_ner_combined = pd.concat([df_pipe_ner, df_joint_ner])
print("NER Comparison:")
print(df_ner_combined.to_string(index=False))

df_re = pd.DataFrame({
    "Model"    : ["Pipeline A", "Joint Model"],
    "Precision": [round(precision_score(re_true_ids, re_pred_ids), 4),
                  round(precision_score(all_re_true, all_re_preds), 4)],
    "Recall"   : [round(recall_score(re_true_ids, re_pred_ids), 4),
                  round(recall_score(all_re_true, all_re_preds), 4)],
    "F1"       : [round(f1_score(re_true_ids, re_pred_ids), 4),
                  round(f1_score(all_re_true, all_re_preds), 4)]
})
print("\nRE Comparison:")
print(df_re.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(3)
for ax, df, title in [
    (axes[0], df_ner_combined[df_ner_combined["Model"]=="Pipeline A"],  "NER — Pipeline A"),
    (axes[1], df_ner_combined[df_ner_combined["Model"]=="Joint Model"], "NER — Joint Model")
]:
    ax.bar(x-0.2, df["Precision"], 0.25, label="Precision", color="#4C72B0")
    ax.bar(x,     df["Recall"],    0.25, label="Recall",    color="#55A868")
    ax.bar(x+0.2, df["F1"],        0.25, label="F1",        color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels(df["Entity"])
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.savefig("ner_comparison.png", dpi=150)
plt.show()

In [ ]:
def categorise_ner_errors(true_seqs, pred_seqs):
    missed = spurious = boundary = type_err = 0
    error_examples = []
    def seq_to_spans(seq):
        spans, start, cur = set(), None, None
        for i, tag in enumerate(seq):
            if tag.startswith('B-'):
                if cur: spans.add((start, i-1, cur))
                start, cur = i, tag[2:]
            elif not tag.startswith('I-'):
                if cur: spans.add((start, i-1, cur))
                start, cur = None, None
        if cur: spans.add((start, len(seq)-1, cur))
        return spans
    for idx, (ts, ps) in enumerate(zip(true_seqs, pred_seqs)):
        gold = seq_to_spans(ts)
        pred = seq_to_spans(ps)
        pred_off = {(s[0],s[1]) for s in pred}
        for g in gold:
            if g in pred: continue
            if (g[0],g[1]) in pred_off: type_err += 1
            elif any(not(p[1]<g[0] or p[0]>g[1]) for p in pred): boundary += 1
            else:
                missed += 1
                if len(error_examples) < 10:
                    error_examples.append(('MISSED', g[2], ts[g[0]:g[1]+1], idx))
        for p in pred:
            if p not in gold and (p[0],p[1]) not in {(g[0],g[1]) for g in gold}:
                if not any(not(g[1]<p[0] or g[0]>p[1]) for g in gold):
                    spurious += 1
                    if len(error_examples) < 10:
                        error_examples.append(('SPURIOUS', p[2], ps[p[0]:p[1]+1], idx))
    return {'Missed': missed, 'Spurious': spurious, 'Boundary': boundary, 'Type': type_err}, error_examples

pipe_errs,  pipe_examples  = categorise_ner_errors(true_tags,    pred_tags)
joint_errs, joint_examples = categorise_ner_errors(all_ner_true, all_ner_preds)

err_df = pd.DataFrame({
    'Error Type': list(pipe_errs.keys()),
    'Pipeline'  : list(pipe_errs.values()),
    'Joint'     : list(joint_errs.values())
})
print("NER ERROR ANALYSIS")
print(err_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(pipe_errs))
w = 0.35
ax.bar(x-w/2, pipe_errs.values(),  w, label='Pipeline', color='#4C72B0')
ax.bar(x+w/2, joint_errs.values(), w, label='Joint',    color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels(pipe_errs.keys())
ax.set_ylabel('Count')
ax.set_title('NER Error Categories — Pipeline vs Joint')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('ner_error_analysis.png', dpi=150)
plt.show()

## 14. Conclusion

This enhanced project implements a full-stack biomedical NLP system:

| Component | Technology | Role |
|-----------|-----------|------|
| **NER** | BioBERT + token classifier | Identify Chemical & Disease entities |
| **RE** | BioBERT + sequence classifier | Classify Chemical-Induced Disease relations |
| **Joint Model** | Shared BioBERT, dual-head | Multi-task NER + RE simultaneously |
| **Vector Store** | ChromaDB + SentenceTransformers | Semantic retrieval over biomedical corpus |
| **RAG** | ChromaDB retriever | Ground LLM answers in indexed evidence |
| **LLM** | Claude / GPT-4 API | Clinical explanation, NER verification, QA |
| **Explainability** | SHAP + LIME + Attention | Interpret model decisions |

### Future Work
- CRF layer on NER head to enforce valid IOB2 transitions
- Hard negative mining for RE class imbalance
- Re-ranking retrieved RAG chunks using cross-encoder
- Knowledge graph integration (MeSH ontology embeddings)
- REST API packaging with Gradio/FastAPI frontend